In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

### DimUser

## Autoloader

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os 
import sys

# print (os.path.join(os.getcwd(),'..','..'))

project_path = os.path.join(os.getcwd(),'..','..')

sys.path.append(project_path)

from utils.transformations import reusable

In [0]:
df_user = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimUser/checkpoint")\
            .option("cloudFiles.schemaEvolutionMode","rescue")\
            .load("abfss://bronze@datalakehaas.dfs.core.windows.net/DimUser")

In [0]:
display(df_user,checkpointLocation = "abfss://silver@datalakehaas.dfs.core.windows.net/DimUser/checkpoint")

Checkpointing to abfss://silver@datalakehaas.dfs.core.windows.net/DimUser/checkpoint


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8888487449054532>, line 1
----> 1 display(df_user,checkpointLocation = "abfss://silver@datalakehaas.dfs.core.windows.net/DimUser/checkpoint")

File <command-6461418600872525>, line 1
----> 1 df_user = spark.readStream.format("cloudFiles")\
      2             .option("cloudFiles.format","parquet")\
      3             .option("cloudFiles.schemaLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimUser/checkpoint")\
      4             .option("cloudFiles.schemaEvolutionMode","rescue")\
      5             .load("abfss://bronze@datalakehaas.dfs.core.windows.net/DimUser")

AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: complete. This output mode is not supported for no streaming aggregations on streaming DataFrames/DataSets. SQLSTATE: 42KDE

JVM stacktrace:
org.apache

In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))


In [0]:
df_user.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
# /Workspace/Users/davidwarnersrh25@gmail.com/SpotifyAzureProject/spotify_dab/utils/transformations.py

# from spotify_dab.utils.transformations import var

# it wont work, coz the root directory is not in sys path



In [0]:
# Add root directory in the system path

# import os 
# import sys

# print (os.path.join(os.getcwd(),'..','..'))

# project_path = os.path.join(os.getcwd(),'..','..')

# sys.path.append(project_path)

# from utils.transformations import var

# print(var)

haas


In [0]:
#create the object
 
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(["user_id"])


In [0]:
df_user.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
        .option("checkpointLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimUser/checkpoint")\
            .trigger(once=True)\
                .option("path","abfss://silver@datalakehaas.dfs.core.windows.net/DimUser/data")\
                .toTable("spotify.silver.DimUser")

### DimArtist

### ReadStream

In [0]:
df_art = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimArtist/checkpoint")\
            .option("cloudFiles.schemaEvolutionMode","rescue")\
            .load("abfss://bronze@datalakehaas.dfs.core.windows.net/DimArtist")

### Transformation

In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art = df_art.dropDuplicates(["artist_id"])

### WriteStream

In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
        .option("checkpointLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimArtist/checkpoint")\
            .trigger(once=True)\
                .option("path","abfss://silver@datalakehaas.dfs.core.windows.net/DimArtist/data")\
                .toTable("spotify.silver.DimArtist")

## DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimTrack/checkpoint")\
            .option("cloudFiles.schemaEvolutionMode","rescue")\
            .load("abfss://bronze@datalakehaas.dfs.core.windows.net/DimTrack")

In [0]:
df_track  = df_track.withColumn("duartionFlag",when(col('duration_sec')<150,"low")\
                                                .when(col("duration_sec")<300,"medium")\
                                                    .otherwise("high"))

df_track = df_track.withColumn('track_name',regexp_replace(col("track_name"),'-',' '))

df_track_obj = reusable()

df_track = df_track_obj.dropColumns(df_track,['_rescued_data'])


In [0]:
df_track.printSchema()

root
 |-- track_id: integer (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_id: integer (nullable = true)
 |-- album_name: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- release_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- duartionFlag: string (nullable = false)



In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
        .option("checkpointLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimTrack/checkpoint")\
            .trigger(once=True)\
                .option("path","abfss://silver@datalakehaas.dfs.core.windows.net/DimTrack/data")\
                .toTable("spotify.silver.DimTrack")

### DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimDate/checkpoint")\
            .option("cloudFiles.schemaEvolutionMode","rescue")\
            .load("abfss://bronze@datalakehaas.dfs.core.windows.net/DimDate")

In [0]:
df_date_obj = reusable()

df_date = df_date_obj.dropColumns(df_date,['_rescued_data'])


In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
        .option("checkpointLocation","abfss://silver@datalakehaas.dfs.core.windows.net/DimDate/checkpoint")\
            .trigger(once=True)\
                .option("path","abfss://silver@datalakehaas.dfs.core.windows.net/DimDate/data")\
                .toTable("spotify.silver.DimDate")

### FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@datalakehaas.dfs.core.windows.net/FactStream/checkpoint")\
            .option("cloudFiles.schemaEvolutionMode","rescue")\
            .load("abfss://bronze@datalakehaas.dfs.core.windows.net/FactStream")

In [0]:
df_fact_obj = reusable()

df_fact = df_fact_obj.dropColumns(df_fact,['_rescued_data'])

In [0]:
df_fact.writeStream.format("delta")\
    .outputMode("append")\
        .option("checkpointLocation","abfss://silver@datalakehaas.dfs.core.windows.net/FactStream/checkpoint")\
            .trigger(once=True)\
                .option("path","abfss://silver@datalakehaas.dfs.core.windows.net/FactStream/data")\
                .toTable("spotify.silver.FactStream")

In [0]:
%sql
select * from spotify.silver.dimuser

user_id,user_name,country,subscription_type,start_date,end_date,updated_at
69,GREGORY HART,Syrian Arab Republic,Premium,2025-01-10,null,2025-10-05T19:49:55.000Z
129,STEPHANIE ROBINSON,Malaysia,Family,2025-04-22,null,2025-10-06T19:49:55.000Z
166,DANIEL RODRIGUEZ,Maldives,Family,2023-10-27,null,2025-09-16T19:49:55.000Z
448,DEANNA MCKINNEY,Vanuatu,Family,2024-05-13,null,2025-09-20T19:49:55.000Z
467,EMILY FRANKLIN,British Indian Ocean Territory (Chagos Archipelago),Premium,2025-05-25,null,2025-09-09T19:49:55.000Z
83,SARAH HINTON,Christmas Island,Free,2023-11-21,null,2025-09-18T19:49:55.000Z
110,RICHARD MARTINEZ,Isle of Man,Free,2023-12-19,null,2025-10-01T19:49:55.000Z
150,ABIGAIL OLSEN,Uruguay,Free,2024-03-04,null,2025-10-07T19:49:55.000Z
369,CASEY CARPENTER,Heard Island and McDonald Islands,Family,2024-07-05,null,2025-10-05T19:49:55.000Z
413,JENNIFER PAYNE,Antarctica (the territory South of 60 deg S),Premium,2025-05-03,null,2025-09-13T19:49:55.000Z
